# PPP Baseline 5.2：一体化可运行版（fastparquet，内存安全，无子进程/虚拟环境）

这版专门修复前面的问题：

- 不再创建虚拟环境，避免 pip/venv 报错；
- 不再用子进程，避免某些 Windows/Jupyter 环境中 `torch` 子进程导入卡死；
- 直接使用当前能跑 `e2e_v5c_eq21_aligned.ipynb` 的 kernel；
- 多 lambda 逐个运行，跑完只保留小型 metrics 表，立即释放完整权重和收益对象，避免 kernel 崩溃；
- MSE-Opt 会单独展示，并和 `e2e_v5c_eq21_aligned.ipynb` 里的 MSE-Opt 参考值做差异检查。

这版 PPP 是 Brandt/Santa-Clara/Valkanov 风格的**线性参数化 baseline**，不是 E2E 优化，也不是神经网络。外部条件和 E2E aligned 文件对齐。


In [1]:
# =========================
# 0. 路径和实验参数
# =========================
from pathlib import Path
import os, sys, subprocess, json, re, time, gc

WINDOWS_DATA_DIR = Path(r"D:\科研和竞赛")
DATA_DIR = WINDOWS_DATA_DIR if WINDOWS_DATA_DIR.exists() else Path("/mnt/data")

LAMBDA_LIST = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
GAMMA = 0.003
MAIN_LAMBDA_FOR_MSE_CHECK = 1.0
PPP_MAXITER = 30
MAX_WINDOWS = None  # debug 可设 2；正式必须 None

print("DATA_DIR =", DATA_DIR)
print("Python =", sys.executable)


DATA_DIR = D:\科研和竞赛
Python = c:\Users\13764\anaconda3\python.exe


In [2]:
# =========================
# 1. 检查依赖：只在缺 fastparquet 时安装 fastparquet
# =========================
# 注意：先 import torch，再 import numpy/pandas/scipy，避免个别环境中 OpenMP/MKL 导入顺序卡死。
try:
    import torch
    print("torch:", torch.__version__)
except Exception as e:
    raise RuntimeError("当前 kernel 缺少 torch。请切换到能运行 e2e_v5c_eq21_aligned.ipynb 的 kernel。") from e

try:
    import numpy as np
    import pandas as pd
    import scipy
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("scipy:", scipy.__version__)
except Exception as e:
    raise RuntimeError("当前 kernel 缺少 numpy/pandas/scipy。请切换到能运行 aligned 文件的 kernel。") from e

try:
    import fastparquet
    print("fastparquet:", fastparquet.__version__)
except Exception:
    print("缺少 fastparquet，开始安装。")
    cmd_mirror = [
        sys.executable, "-m", "pip", "install", "fastparquet",
        "--index-url", "https://pypi.tuna.tsinghua.edu.cn/simple",
        "--trusted-host", "pypi.tuna.tsinghua.edu.cn",
        "--prefer-binary", "--no-input", "--timeout", "120", "--retries", "3",
    ]
    rc = subprocess.call(cmd_mirror)
    if rc != 0:
        print("清华源失败，尝试默认 PyPI。")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "fastparquet", "--prefer-binary", "--no-input", "--timeout", "120", "--retries", "3"])
    import fastparquet
    print("fastparquet installed:", fastparquet.__version__)


torch: 2.11.0+cpu
numpy: 2.1.3
pandas: 2.2.3
scipy: 1.15.3
fastparquet: 2026.3.0


In [3]:
# =========================
# 2. 定义 PPP/MSE 所需全部函数和类
# =========================
# 下面是核心 runner 代码，已移除命令行 main，直接在 notebook kernel 中定义函数。
# -*- coding: utf-8 -*-
"""
PPP baseline runner aligned with e2e_v5c_eq21_aligned.ipynb.

Design goals:
1. MSE-Opt is a reference baseline copied/aligned with e2e_v5c_eq21_aligned:
   - same MLP architecture
   - same 70 MSE epochs = N_WARM + N_E2E
   - same rolling window and seed schedule
   - same downstream SOCP+TC optimizer
   - same realized transaction cost: 0.5 * gamma * ||dw||_1
2. PPP is a Brandt-style linear parametric portfolio policy baseline, not an E2E optimizer.
3. This file uses fastparquet only for parquet reading.
4. For Jupyter stability, run lambdas in separate subprocesses from the notebook.
"""

import os
import sys
import time
import gc
import json
import argparse
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# Import torch before scipy/numpy stack to avoid occasional OpenMP/MKL deadlocks in some Jupyter kernels.
import torch
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize

# ============================
# Aligned hyperparameters
# ============================
IC_TSTAT_THRESHOLD = 2.0
MIN_COVERAGE       = 0.70
FWD_RET_COVERAGE   = 0.80
HIDDEN_DIM         = 32
SEED               = 42

N_WARM     = 10
N_E2E      = 60
LR_MSE     = 1e-3
GRAD_CLIP  = 1.0
BATCH_SIZE = 8

GAMMA       = 0.003
EPS_TC      = 1e-4
MAX_W_FRAC  = 10
TRAIN_WINDOW = 36
STEP_SIZE    = 3

LAMBDA_LIST_DEFAULT = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

# PPP optimization controls. These only affect the PPP baseline.
PPP_L2      = 1e-3
PPP_MAXITER = 120

# Reference line from e2e_v5c_eq21_aligned.ipynb screenshot/output.
# This is printed for alignment checking; the runner still computes MSE-Opt from scratch.
REFERENCE_MSE = {
    "lambda": 1.0,
    "gamma": 0.003,
    "Ann_Ret": 0.1803,
    "Ann_Vol": 0.2345,
    "Sharpe": 0.769,
    "Sortino": 2.145,
    "MDD": -0.1881,
    "Win": 0.467,
    "AvgTO": 1.3271,
}


def log(msg: str) -> None:
    print(msg, flush=True)


def require_fastparquet():
    try:
        import fastparquet  # noqa: F401
    except Exception as e:
        raise RuntimeError(
            "当前环境没有 fastparquet。请先在虚拟环境里安装：\n"
            "  python -m pip install fastparquet -i https://pypi.tuna.tsinghua.edu.cn/simple\n"
            "然后在 Jupyter 中选择该虚拟环境对应的 kernel。"
        ) from e


def resolve_paths(data_dir: str):
    data_dir = Path(data_dir)
    raw_candidates = [
        data_dir / "hs300_factor_data_filled.parquet",
        data_dir / "hs300_factor_data_filled(1).parquet",
        Path("/mnt/data/hs300_factor_data_filled(1).parquet"),
    ]
    panel_candidates = [
        data_dir / "factor_panel_v2_processed.parquet",
        data_dir / "factor_panel_v2_processed(1).parquet",
        Path("/mnt/data/factor_panel_v2_processed(1).parquet"),
    ]
    ic_candidates = [
        data_dir / "03v3_ic_summary.csv",
        data_dir / "03v3_ic_summary(2).csv",
        Path("/mnt/data/03v3_ic_summary(2).csv"),
        Path("/mnt/data/03v3_ic_summary.csv"),
    ]

    def pick(cands, name):
        for p in cands:
            if p.exists():
                return str(p)
        raise FileNotFoundError(f"找不到 {name}，候选路径：\n" + "\n".join(map(str, cands)))

    return pick(raw_candidates, "RAW_PATH"), pick(panel_candidates, "PANEL_PATH"), pick(ic_candidates, "IC_PATH")


class DataBundle:
    pass


def load_data(data_dir: str) -> DataBundle:
    require_fastparquet()
    raw_path, panel_path, ic_path = resolve_paths(data_dir)

    log("=" * 80)
    log("Loading data with fastparquet")
    log(f"RAW_PATH   = {raw_path}")
    log(f"PANEL_PATH = {panel_path}")
    log(f"IC_PATH    = {ic_path}")
    log("=" * 80)

    panel = pd.read_parquet(panel_path, engine="fastparquet")
    if "date" not in panel.columns:
        panel = panel.reset_index()
    panel["date"] = pd.to_datetime(panel["date"])

    ic_df = pd.read_csv(ic_path, index_col=0)
    if "Sig(|t|>2)" in ic_df.columns:
        sig_mask = ic_df["Sig(|t|>2)"].astype(bool)
    elif "t_stat" in ic_df.columns:
        sig_mask = ic_df["t_stat"].abs() > IC_TSTAT_THRESHOLD
    else:
        raise ValueError("IC 文件需含 t_stat 或 Sig(|t|>2) 列。")

    sig_factors = list(ic_df[sig_mask].index)
    n_features = len(sig_factors)
    if n_features < 2:
        raise ValueError("显著因子数量过少，请检查 IC 文件。")
    if "fwd_ret" not in panel.columns:
        raise ValueError("panel 文件必须包含 fwd_ret。")
    if "ts_code" not in panel.columns or "date" not in panel.columns:
        raise ValueError("panel 文件必须包含 ts_code 和 date。")

    all_dates = sorted(panel["date"].unique())

    def get_valid_stocks(df, factors):
        return set(df.dropna(subset=factors + ["fwd_ret"])["ts_code"].unique())

    month_valid = {d: get_valid_stocks(panel[panel["date"] == d], sig_factors) for d in all_dates}
    stock_cnt = Counter(s for v in month_valid.values() for s in v)
    universe = sorted(s for s, c in stock_cnt.items() if c / len(all_dates) >= MIN_COVERAGE)
    n = len(universe)
    max_w = MAX_W_FRAC / n

    panel_u = panel[panel["ts_code"].isin(universe)].sort_values(["date", "ts_code"]).reset_index(drop=True)
    for col in sig_factors:
        panel_u[col] = panel_u.groupby("date")[col].transform(
            lambda x: x.fillna(0 if x.isna().all() else x.mean())
        )
    panel_u["fwd_ret"] = panel_u.groupby("date")["fwd_ret"].transform(
        lambda x: x.fillna(x.median() if not x.isna().all() else 0)
    )

    def cross_sec_zscore(arr):
        mu = arr.mean(0, keepdims=True)
        sig = np.maximum(arr.std(0, keepdims=True), 1e-8)
        return np.nan_to_num((arr - mu) / sig).astype("float32")

    fwd_cov = panel_u.groupby("date")["fwd_ret"].apply(lambda x: x.notna().mean())
    dates_valid = [d for d in sorted(panel_u["date"].unique()) if fwd_cov.get(d, 0) >= FWD_RET_COVERAGE]

    x_tensors, y_tensors, x_np, y_np = [], [], [], []
    for d in dates_valid:
        sub = panel_u[panel_u["date"] == d].sort_values("ts_code")
        x = cross_sec_zscore(sub[sig_factors].values)
        y = sub["fwd_ret"].values.astype("float32")
        x_tensors.append(torch.tensor(x))
        y_tensors.append(torch.tensor(y))
        x_np.append(x)
        y_np.append(y)

    rolling_windows = []
    start = 0
    t_total = len(x_tensors)
    while start + TRAIN_WINDOW + STEP_SIZE <= t_total:
        rolling_windows.append(
            {
                "train_start": start,
                "train_end": start + TRAIN_WINDOW,
                "test_start": start + TRAIN_WINDOW,
                "test_end": min(start + TRAIN_WINDOW + STEP_SIZE, t_total),
            }
        )
        start += STEP_SIZE

    if not rolling_windows:
        raise ValueError("滚动窗口为空，请检查有效月份数量。")

    db = DataBundle()
    db.raw_path, db.panel_path, db.ic_path = raw_path, panel_path, ic_path
    db.panel = panel
    db.sig_factors = sig_factors
    db.n_features = n_features
    db.universe = universe
    db.n = n
    db.max_w = max_w
    db.dates_valid = dates_valid
    db.x_tensors = x_tensors
    db.y_tensors = y_tensors
    db.x_np = x_np
    db.y_np = y_np
    db.rolling_windows = rolling_windows

    fw, lw = rolling_windows[0], rolling_windows[-1]
    log(f"Factor panel: {panel.shape}  ({panel['date'].nunique()} months)")
    log(f"日期范围: {panel['date'].min().date()} ~ {panel['date'].max().date()}")
    log(f"显著因子 ({n_features} 个): {sig_factors}")
    log(f"股票池: {n} 只   max_w = {MAX_W_FRAC}/N = {max_w:.6f}  有效月份: {t_total}")
    log(f"滚动窗口: {len(rolling_windows)} 个")
    log(
        f"首个测试期: {pd.Timestamp(dates_valid[fw['test_start']]).strftime('%Y-%m')} ~ "
        f"{pd.Timestamp(dates_valid[fw['test_end']-1]).strftime('%Y-%m')}"
    )
    log(
        f"最后测试期: {pd.Timestamp(dates_valid[lw['test_start']]).strftime('%Y-%m')} ~ "
        f"{pd.Timestamp(dates_valid[lw['test_end']-1]).strftime('%Y-%m')}"
    )
    return db


# ============================
# Shared optimizer and metrics
# ============================

def smooth_l1_grad_vec(delta, eps):
    return delta / np.sqrt(delta ** 2 + eps)


def solve_socp_tc(mu_np, var_np, w_prev_np, lambda_, gamma, max_w, zeta, eps_tc=1e-4,
                  max_outer=30, max_inner=50, tol=1e-10):
    """Forward solver copied from the aligned E2E notebook's downstream optimizer."""
    n = len(mu_np)
    z0 = max(zeta, 1e-6)

    def init_budget(nu):
        return np.clip((mu_np - nu) / z0, 0, max_w).sum() - 1.0

    try:
        nu0 = brentq(init_budget, mu_np.min() - max_w * z0 - 1.0, mu_np.max() + 1.0,
                     xtol=1e-12, maxiter=300)
        w = np.clip((mu_np - nu0) / z0, 0, max_w)
        s = w.sum()
        w = w / s if s > 1e-12 else np.ones(n) / n
    except Exception:
        w = np.ones(n) / n

    P = max(np.sqrt((w ** 2 * var_np).sum()), 1e-8)

    for _ in range(max_outer):
        delta = w - w_prev_np
        tc_grad = 0.5 * gamma * smooth_l1_grad_vec(delta, eps_tc)
        mu_eff = mu_np - tc_grad

        w_new, P_new = w.copy(), P
        for _ in range(max_inner):
            eff_z = lambda_ * var_np / P_new + zeta

            def budget(nu):
                return np.clip((mu_eff - nu) / eff_z, 0, max_w).sum() - 1.0

            nu_lo = (mu_eff - eff_z * max_w).max() - 1.0
            nu_hi = mu_eff.max() + 1.0

            try:
                nu = brentq(budget, nu_lo, nu_hi, xtol=1e-12, maxiter=400)
                w_try = np.clip((mu_eff - nu) / eff_z, 0, max_w)
                s = w_try.sum()
                w_try = w_try / s if s > 1e-12 else np.ones(n) / n
            except Exception:
                break

            P_try = max(np.sqrt((w_try ** 2 * var_np).sum()), 1e-8)
            if abs(P_try - P_new) < tol:
                w_new, P_new = w_try, P_try
                break
            w_new, P_new = w_try, P_try

        if np.max(np.abs(w_new - w)) < tol:
            w, P = w_new, P_new
            break
        w, P = w_new, P_new

    return w, P


def compute_var_vec(y_tr_list):
    mat = np.stack([y.numpy() if hasattr(y, "numpy") else y for y in y_tr_list])
    return np.maximum(mat.var(axis=0), 1e-6).astype("float32")


def calc_turnover(weights_list, n):
    to = []
    w_prev = np.ones(n) / n
    for w in weights_list:
        w = np.asarray(w, dtype=float)
        to.append(np.abs(w - w_prev).sum())
        w_prev = w
    return np.asarray(to)


def metrics(returns, weights_list, n, label="", verbose=True):
    r = np.asarray(returns, dtype=float)
    ann_ret = r.mean() * 12
    ann_vol = r.std(ddof=1) * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0.0
    cum = np.cumprod(1 + r)
    dd = (cum / np.maximum.accumulate(cum)) - 1
    neg = r[r < 0]
    sortino = ann_ret / (neg.std(ddof=1) * np.sqrt(12)) if len(neg) > 1 and neg.std(ddof=1) > 0 else np.nan
    win = (r > 0).mean()
    avg_to = calc_turnover(weights_list, n).mean() if weights_list is not None else 0.0
    out = {
        "Ann_Ret": float(ann_ret),
        "Ann_Vol": float(ann_vol),
        "Sharpe": float(sharpe),
        "Sortino": float(sortino),
        "MDD": float(dd.min()),
        "Win": float(win),
        "AvgTO": float(avg_to),
    }
    if verbose:
        log(format_metrics(label, out))
    return out


def format_metrics(label, m):
    return (
        f"{label:<28} "
        f"AnnRet={m['Ann_Ret']:+.2%}  "
        f"AnnVol={m['Ann_Vol']:.2%}  "
        f"SR={m['Sharpe']:.3f}  "
        f"Sortino={m['Sortino']:.3f}  "
        f"MDD={m['MDD']:.2%}  "
        f"Win={m['Win']:.1%}  "
        f"AvgTO(two-sided)={m['AvgTO']:.2%}"
    )


def print_mse_alignment(m):
    ref = REFERENCE_MSE
    log("\nAligned notebook reference line for MSE-Opt:")
    log(format_metrics(f"MSE-Opt REF (l={ref['lambda']}, g={ref['gamma']})", ref))
    log("Difference: this runner - aligned reference")
    log(
        f"AnnRet {m['Ann_Ret'] - ref['Ann_Ret']:+.2%} | "
        f"AnnVol {m['Ann_Vol'] - ref['Ann_Vol']:+.2%} | "
        f"SR {m['Sharpe'] - ref['Sharpe']:+.3f} | "
        f"Sortino {m['Sortino'] - ref['Sortino']:+.3f} | "
        f"MDD {m['MDD'] - ref['MDD']:+.2%} | "
        f"Win {m['Win'] - ref['Win']:+.1%} | "
        f"AvgTO {m['AvgTO'] - ref['AvgTO']:+.2%}"
    )
    log("说明：如果这里差异明显，优先检查数据文件、路径、随机种子、窗口划分或依赖环境。")


# ============================
# MSE-Opt baseline
# ============================
class CrossSectionalMLP(nn.Module):
    def __init__(self, n_features, hidden=HIDDEN_DIM):
        super().__init__()
        h2 = max(hidden // 2, 4)
        self.net = nn.Sequential(
            nn.LayerNorm(n_features),
            nn.Linear(n_features, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.05),
            nn.Linear(hidden, h2),         nn.BatchNorm1d(h2),     nn.ReLU(), nn.Dropout(0.05),
            nn.Linear(h2, 1),
        )

    def forward(self, X):
        return self.net(X).squeeze(-1)


def train_mse_v5c(model, x_tr, y_tr, n_epochs, lr):
    t_tr = len(x_tr)
    hist = []
    opt = optim.Adam(model.parameters(), lr=lr)
    for ep in range(n_epochs):
        model.train()
        ep_loss = 0.0
        n_batches = 0
        for bs in range(0, t_tr, BATCH_SIZE):
            be = min(bs + BATCH_SIZE, t_tr)
            opt.zero_grad()
            bl = sum(F.mse_loss(model(x_tr[t]), y_tr[t]) for t in range(bs, be))
            avg = bl / (be - bs)
            avg.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            ep_loss += avg.item()
            n_batches += 1
        hist.append(ep_loss / max(n_batches, 1))
    return hist


def run_mse_opt(db: DataBundle, lambda_: float, gamma: float, max_windows=None):
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    res = {"returns": [], "gross_returns": [], "tc_costs": [], "weights": [], "dates": [], "bench": []}
    w_prev = np.ones(db.n, dtype=np.float64) / db.n
    windows = db.rolling_windows[:max_windows] if max_windows else db.rolling_windows
    t0 = time.time()

    for idx, win in enumerate(windows):
        ts, te = win["train_start"], win["train_end"]
        vs, ve = win["test_start"], win["test_end"]
        x_tr = [db.x_tensors[i] for i in range(ts, te)]
        y_tr = [db.y_tensors[i] for i in range(ts, te)]
        var_vec = compute_var_vec(y_tr).astype(np.float64)

        torch.manual_seed(SEED + idx * 37)
        model = CrossSectionalMLP(db.n_features)
        hist = train_mse_v5c(model, x_tr, y_tr, N_WARM + N_E2E, LR_MSE)
        model.eval()

        for i in range(vs, ve):
            with torch.no_grad():
                mu_hat = model(db.x_tensors[i]).detach().cpu().numpy().astype(np.float64)
            w, _ = solve_socp_tc(mu_hat, var_vec, w_prev, lambda_, gamma, db.max_w, zeta=0.0, eps_tc=EPS_TC)
            y = db.y_np[i].astype(np.float64)
            gross = float(np.dot(w, y))
            tc = float(0.5 * gamma * np.abs(w - w_prev).sum())
            net = gross - tc
            res["returns"].append(net)
            res["gross_returns"].append(gross)
            res["tc_costs"].append(tc)
            res["weights"].append(w.astype("float32"))
            res["dates"].append(db.dates_valid[i])
            res["bench"].append(float(np.mean(y)))
            w_prev = w

        log(
            f"[MSE-Opt | lambda={lambda_} | gamma={gamma}] "
            f"win {idx+1:02d}/{len(windows)}  "
            f"test {pd.Timestamp(db.dates_valid[vs]).strftime('%y-%m')}~{pd.Timestamp(db.dates_valid[ve-1]).strftime('%y-%m')}  "
            f"loss={hist[-1]:+.4f}  elapsed={(time.time()-t0)/60:.1f}min"
        )
        del model, hist, x_tr, y_tr
        gc.collect()
    return res


# ============================
# Brandt-style linear PPP baseline
# ============================
def sigmoid_stable(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def ppp_weights_from_theta(x, theta, max_w):
    """Capped logistic mapping: 0 <= w_i <= max_w and sum_i w_i = 1."""
    s = np.asarray(x, dtype=np.float64) @ np.asarray(theta, dtype=np.float64)
    lo = float(s.min() - 60.0)
    hi = float(s.max() + 60.0)

    def budget(tau):
        return max_w * sigmoid_stable(s - tau).sum() - 1.0

    try:
        tau = brentq(budget, lo, hi, xtol=1e-12, maxiter=100)
        w = max_w * sigmoid_stable(s - tau)
        w = w / w.sum()
        w = np.minimum(np.maximum(w, 0.0), max_w)
        # numerical cleanup after clipping
        if abs(w.sum() - 1.0) > 1e-10:
            w = project_capped_simplex(w, max_w)
        return w.astype(np.float64)
    except Exception:
        z = np.exp(np.clip(s - s.max(), -50, 50))
        w = z / z.sum()
        return project_capped_simplex(w, max_w).astype(np.float64)


def project_capped_simplex(v, max_w, max_iter=50):
    """Simple projection-like repair for numerical fallback; exact enough for emergency use."""
    w = np.maximum(np.asarray(v, dtype=np.float64), 0.0)
    if w.sum() <= 1e-14:
        return np.ones_like(w) / len(w)
    w = w / w.sum()
    for _ in range(max_iter):
        over = w > max_w
        if not over.any():
            break
        fixed_mass = max_w * over.sum()
        rem = 1.0 - fixed_mass
        w[over] = max_w
        if rem <= 0:
            break
        under = ~over
        s = w[under].sum()
        if s <= 1e-14:
            w[under] = rem / under.sum()
        else:
            w[under] = w[under] / s * rem
    return w


def ppp_objective(theta, x_list, y_list, var_vec, n, max_w, lambda_, gamma, reg=PPP_L2):
    w_prev = np.ones(n, dtype=np.float64) / n
    loss = 0.0
    theta = np.asarray(theta, dtype=np.float64)
    for x, y in zip(x_list, y_list):
        w = ppp_weights_from_theta(x, theta, max_w)
        y = y.astype(np.float64)
        port_ret = float(np.dot(w, y))
        tc = float(0.5 * gamma * np.abs(w - w_prev).sum())
        risk = float(lambda_ * np.sqrt(np.dot(w * w, var_vec) + 1e-12))
        loss += -port_ret + tc + risk
        w_prev = w
    return loss / len(x_list) + 0.5 * reg * float(np.dot(theta, theta))


def train_ppp_theta(db, x_list, y_list, var_vec, lambda_, gamma, theta0=None, maxiter=PPP_MAXITER):
    if theta0 is None:
        theta0 = np.zeros(db.n_features, dtype=np.float64)
    opt = minimize(
        ppp_objective,
        theta0,
        args=(x_list, y_list, var_vec, db.n, db.max_w, lambda_, gamma, PPP_L2),
        method="Powell",
        options={"maxiter": int(maxiter), "xtol": 1e-5, "ftol": 1e-7, "disp": False},
    )
    return np.asarray(opt.x, dtype=np.float64), opt


def run_ppp(db: DataBundle, lambda_: float, gamma: float, max_windows=None, maxiter=PPP_MAXITER):
    res = {"returns": [], "gross_returns": [], "tc_costs": [], "weights": [], "dates": [], "bench": [], "theta": []}
    w_prev = np.ones(db.n, dtype=np.float64) / db.n
    theta_prev = np.zeros(db.n_features, dtype=np.float64)
    windows = db.rolling_windows[:max_windows] if max_windows else db.rolling_windows
    t0 = time.time()

    for idx, win in enumerate(windows):
        ts, te = win["train_start"], win["train_end"]
        vs, ve = win["test_start"], win["test_end"]
        x_tr = [db.x_np[i] for i in range(ts, te)]
        y_tr = [db.y_np[i] for i in range(ts, te)]
        var_vec = np.maximum(np.stack(y_tr).var(axis=0), 1e-6).astype(np.float64)

        theta, opt = train_ppp_theta(db, x_tr, y_tr, var_vec, lambda_, gamma, theta0=theta_prev, maxiter=maxiter)
        theta_prev = theta
        res["theta"].append(theta.astype("float32"))

        for i in range(vs, ve):
            w = ppp_weights_from_theta(db.x_np[i], theta, db.max_w)
            y = db.y_np[i].astype(np.float64)
            gross = float(np.dot(w, y))
            tc = float(0.5 * gamma * np.abs(w - w_prev).sum())
            net = gross - tc
            res["returns"].append(net)
            res["gross_returns"].append(gross)
            res["tc_costs"].append(tc)
            res["weights"].append(w.astype("float32"))
            res["dates"].append(db.dates_valid[i])
            res["bench"].append(float(np.mean(y)))
            w_prev = w

        log(
            f"[PPP linear | lambda={lambda_} | gamma={gamma}] "
            f"win {idx+1:02d}/{len(windows)}  "
            f"test {pd.Timestamp(db.dates_valid[vs]).strftime('%y-%m')}~{pd.Timestamp(db.dates_valid[ve-1]).strftime('%y-%m')}  "
            f"obj={opt.fun:+.6f}  success={opt.success}  elapsed={(time.time()-t0)/60:.1f}min"
        )
        del x_tr, y_tr, var_vec, opt
        gc.collect()
    return res


def run_mode(args):
    log(f"Python executable: {sys.executable}")
    log(f"pandas={pd.__version__}, torch={torch.__version__}")
    import fastparquet
    log(f"fastparquet={fastparquet.__version__}")
    log(f"Hyperparams: train_window={TRAIN_WINDOW}, step={STEP_SIZE}, gamma={args.gamma}, lambda={args.lambda_}")
    db = load_data(args.data_dir)

    if args.mode == "check":
        return

    if args.mode == "mse":
        log("\n" + "=" * 86)
        log("Running MSE-Opt reference baseline. This should match e2e_v5c_eq21_aligned.ipynb MSE-Opt.")
        log("=" * 86)
        res_mse = run_mse_opt(db, args.lambda_, args.gamma, max_windows=args.max_windows)
        m_mse = metrics(res_mse["returns"], res_mse["weights"], db.n, f"MSE-Opt (l={args.lambda_}, g={args.gamma})")
        if args.max_windows is None and abs(args.lambda_ - 1.0) < 1e-12 and abs(args.gamma - 0.003) < 1e-12:
            print_mse_alignment(m_mse)
        m_bench = metrics(np.asarray(res_mse["bench"]), None, db.n, "Equal-Weight")
        log("JSON_RESULT " + json.dumps({"method": "MSE-Opt", "lambda": args.lambda_, "gamma": args.gamma, "metrics": m_mse}, ensure_ascii=False))
        return

    if args.mode == "ppp":
        log("\n" + "=" * 86)
        log("Running Brandt-style linear PPP baseline. No E2E optimization layer is used.")
        log("=" * 86)
        res_ppp = run_ppp(db, args.lambda_, args.gamma, max_windows=args.max_windows, maxiter=args.ppp_maxiter)
        m_ppp = metrics(res_ppp["returns"], res_ppp["weights"], db.n, f"PPP linear (l={args.lambda_}, g={args.gamma})")
        m_bench = metrics(np.asarray(res_ppp["bench"]), None, db.n, "Equal-Weight")
        log("JSON_RESULT " + json.dumps({"method": "PPP", "lambda": args.lambda_, "gamma": args.gamma, "metrics": m_ppp}, ensure_ascii=False))
        return

    raise ValueError(f"Unsupported mode: {args.mode}")


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--mode", choices=["check", "mse", "ppp"], default="check")
    p.add_argument("--data-dir", default=r"D:\科研和竞赛")
    p.add_argument("--lambda", dest="lambda_", type=float, default=1.0)
    p.add_argument("--gamma", type=float, default=GAMMA)
    p.add_argument("--max-windows", type=int, default=None, help="debug only; omit for full run")
    p.add_argument("--ppp-maxiter", type=int, default=PPP_MAXITER)
    return p.parse_args(argv)


print('核心函数已加载。')
print('TRAIN_WINDOW =', TRAIN_WINDOW, 'STEP_SIZE =', STEP_SIZE, 'N_WARM+N_E2E =', N_WARM + N_E2E)


核心函数已加载。
TRAIN_WINDOW = 36 STEP_SIZE = 3 N_WARM+N_E2E = 70


In [4]:
# =========================
# 3. 读取数据并检查窗口
# =========================
db = load_data(str(DATA_DIR))
print("\n数据读取完成：")
print("n =", db.n, "n_features =", db.n_features)
print("months =", len(db.dates_valid), "rolling windows =", len(db.rolling_windows))
print("first test:", db.dates_valid[db.rolling_windows[0]['test_start']], "~", db.dates_valid[db.rolling_windows[0]['test_end']-1])
print("last  test:", db.dates_valid[db.rolling_windows[-1]['test_start']], "~", db.dates_valid[db.rolling_windows[-1]['test_end']-1])


Loading data with fastparquet
RAW_PATH   = D:\科研和竞赛\hs300_factor_data_filled.parquet
PANEL_PATH = D:\科研和竞赛\factor_panel_v2_processed.parquet
IC_PATH    = D:\科研和竞赛\03v3_ic_summary.csv
Factor panel: (21760, 105)  (68 months)
日期范围: 2020-04-30 ~ 2025-11-28
显著因子 (10 个): ['F91_ocf_ni', 'F97_accruals', 'F80_ocf_mv', 'F74_cfp', 'F55_holding_adj', 'F81_size', 'F34_vol_of_vol', 'F88_cash_roe', 'F41_turn_std', 'F30_range_vol']
股票池: 252 只   max_w = 10/N = 0.039683  有效月份: 68
滚动窗口: 10 个
首个测试期: 2023-04 ~ 2023-06
最后测试期: 2025-07 ~ 2025-09

数据读取完成：
n = 252 n_features = 10
months = 68 rolling windows = 10
first test: 2023-04-28 00:00:00 ~ 2023-06-30 00:00:00
last  test: 2025-07-31 00:00:00 ~ 2025-09-30 00:00:00


In [5]:
# =========================
# 4. 跑 MSE-Opt reference baseline，并展示与 aligned 文件的对齐
# =========================
print("理论参考值来自 e2e_v5c_eq21_aligned.ipynb 中展示的 MSE-Opt：")
print("MSE-Opt (l=1.0, g=0.003) AnnRet=+18.03%  AnnVol=23.45%  SR=0.769  Sortino=2.145  MDD=-18.81%  Win=46.7%  AvgTO(two-sided)=132.71%")
print("\n现在重新运行本 notebook 的 MSE-Opt。理论上应该和上面几乎一样；否则说明外部设置没有对齐。")

res_mse = run_mse_opt(db, MAIN_LAMBDA_FOR_MSE_CHECK, GAMMA, max_windows=MAX_WINDOWS)
m_mse = metrics(res_mse["returns"], res_mse["weights"], db.n, f"MSE-Opt (l={MAIN_LAMBDA_FOR_MSE_CHECK}, g={GAMMA})")
print_mse_alignment(m_mse)

mse_df = pd.DataFrame([{**{"method": "MSE-Opt", "lambda": MAIN_LAMBDA_FOR_MSE_CHECK, "gamma": GAMMA}, **m_mse}])
display(mse_df)

del res_mse
gc.collect()


理论参考值来自 e2e_v5c_eq21_aligned.ipynb 中展示的 MSE-Opt：
MSE-Opt (l=1.0, g=0.003) AnnRet=+18.03%  AnnVol=23.45%  SR=0.769  Sortino=2.145  MDD=-18.81%  Win=46.7%  AvgTO(two-sided)=132.71%

现在重新运行本 notebook 的 MSE-Opt。理论上应该和上面几乎一样；否则说明外部设置没有对齐。
[MSE-Opt | lambda=1.0 | gamma=0.003] win 01/10  test 23-04~23-06  loss=+0.0218  elapsed=0.1min
[MSE-Opt | lambda=1.0 | gamma=0.003] win 02/10  test 23-07~23-09  loss=+0.0175  elapsed=0.2min
[MSE-Opt | lambda=1.0 | gamma=0.003] win 03/10  test 23-10~23-12  loss=+0.0180  elapsed=0.2min
[MSE-Opt | lambda=1.0 | gamma=0.003] win 04/10  test 24-01~24-03  loss=+0.0160  elapsed=0.3min
[MSE-Opt | lambda=1.0 | gamma=0.003] win 05/10  test 24-04~24-06  loss=+0.0162  elapsed=0.3min
[MSE-Opt | lambda=1.0 | gamma=0.003] win 06/10  test 24-07~24-09  loss=+0.0177  elapsed=0.4min
[MSE-Opt | lambda=1.0 | gamma=0.003] win 07/10  test 24-10~24-12  loss=+0.0161  elapsed=0.5min
[MSE-Opt | lambda=1.0 | gamma=0.003] win 08/10  test 25-01~25-03  loss=+0.0144  elapsed=0.6min
[MSE-O

,method,lambda,gamma,Ann_Ret,Ann_Vol,Sharpe,Sortino,MDD,Win,AvgTO
0,MSE-Opt,1.0,0.003,0.180694,0.234559,0.770353,2.150754,-0.188228,0.466667,1.327022


0

In [6]:
# =========================
# 5. 跑 PPP baseline：多 lambda 逐个运行、逐个展示，不保存大对象
# =========================
ppp_rows = []
for lam in LAMBDA_LIST:
    print("\n" + "=" * 100)
    print(f"Running PPP linear baseline: lambda={lam}, gamma={GAMMA}")
    print("=" * 100)
    res_ppp = run_ppp(db, lam, GAMMA, max_windows=MAX_WINDOWS, maxiter=PPP_MAXITER)
    m_ppp = metrics(res_ppp["returns"], res_ppp["weights"], db.n, f"PPP linear (l={lam}, g={GAMMA})")
    row = {**{"method": "PPP", "lambda": lam, "gamma": GAMMA}, **m_ppp}
    ppp_rows.append(row)
    display(pd.DataFrame([row]))
    del res_ppp
    gc.collect()

ppp_summary = pd.DataFrame(ppp_rows)
print("\nPPP lambda sweep summary:")
display(ppp_summary)



Running PPP linear baseline: lambda=0.1, gamma=0.003
[PPP linear | lambda=0.1 | gamma=0.003] win 01/10  test 23-04~23-06  obj=-0.039017  success=True  elapsed=0.3min
[PPP linear | lambda=0.1 | gamma=0.003] win 02/10  test 23-07~23-09  obj=-0.026849  success=True  elapsed=0.8min
[PPP linear | lambda=0.1 | gamma=0.003] win 03/10  test 23-10~23-12  obj=-0.023785  success=True  elapsed=1.3min
[PPP linear | lambda=0.1 | gamma=0.003] win 04/10  test 24-01~24-03  obj=-0.015355  success=True  elapsed=1.8min
[PPP linear | lambda=0.1 | gamma=0.003] win 05/10  test 24-04~24-06  obj=-0.019736  success=True  elapsed=2.4min
[PPP linear | lambda=0.1 | gamma=0.003] win 06/10  test 24-07~24-09  obj=-0.017027  success=True  elapsed=2.9min
[PPP linear | lambda=0.1 | gamma=0.003] win 07/10  test 24-10~24-12  obj=-0.022759  success=True  elapsed=3.2min
[PPP linear | lambda=0.1 | gamma=0.003] win 08/10  test 25-01~25-03  obj=-0.020132  success=True  elapsed=3.6min
[PPP linear | lambda=0.1 | gamma=0.003] wi

,method,lambda,gamma,Ann_Ret,Ann_Vol,Sharpe,Sortino,MDD,Win,AvgTO
0,PPP,0.1,0.003,0.297382,0.282384,1.053111,2.274627,-0.186816,0.6,0.635446



Running PPP linear baseline: lambda=0.5, gamma=0.003
[PPP linear | lambda=0.5 | gamma=0.003] win 01/10  test 23-04~23-06  obj=-0.029730  success=True  elapsed=0.3min
[PPP linear | lambda=0.5 | gamma=0.003] win 02/10  test 23-07~23-09  obj=-0.018386  success=True  elapsed=0.6min
[PPP linear | lambda=0.5 | gamma=0.003] win 03/10  test 23-10~23-12  obj=-0.015674  success=True  elapsed=0.9min
[PPP linear | lambda=0.5 | gamma=0.003] win 04/10  test 24-01~24-03  obj=-0.008008  success=True  elapsed=1.3min
[PPP linear | lambda=0.5 | gamma=0.003] win 05/10  test 24-04~24-06  obj=-0.011499  success=True  elapsed=1.7min
[PPP linear | lambda=0.5 | gamma=0.003] win 06/10  test 24-07~24-09  obj=-0.009883  success=True  elapsed=2.0min
[PPP linear | lambda=0.5 | gamma=0.003] win 07/10  test 24-10~24-12  obj=-0.014454  success=True  elapsed=2.3min
[PPP linear | lambda=0.5 | gamma=0.003] win 08/10  test 25-01~25-03  obj=-0.012175  success=True  elapsed=2.5min
[PPP linear | lambda=0.5 | gamma=0.003] wi

,method,lambda,gamma,Ann_Ret,Ann_Vol,Sharpe,Sortino,MDD,Win,AvgTO
0,PPP,0.5,0.003,0.272896,0.264275,1.032618,2.265719,-0.17658,0.6,0.590013



Running PPP linear baseline: lambda=1.0, gamma=0.003
[PPP linear | lambda=1.0 | gamma=0.003] win 01/10  test 23-04~23-06  obj=-0.019251  success=True  elapsed=0.2min
[PPP linear | lambda=1.0 | gamma=0.003] win 02/10  test 23-07~23-09  obj=-0.009280  success=True  elapsed=0.5min
[PPP linear | lambda=1.0 | gamma=0.003] win 03/10  test 23-10~23-12  obj=-0.007011  success=True  elapsed=0.8min
[PPP linear | lambda=1.0 | gamma=0.003] win 04/10  test 24-01~24-03  obj=-0.000209  success=True  elapsed=1.1min
[PPP linear | lambda=1.0 | gamma=0.003] win 05/10  test 24-04~24-06  obj=-0.003022  success=True  elapsed=1.4min
[PPP linear | lambda=1.0 | gamma=0.003] win 06/10  test 24-07~24-09  obj=-0.002517  success=True  elapsed=1.7min
[PPP linear | lambda=1.0 | gamma=0.003] win 07/10  test 24-10~24-12  obj=-0.005340  success=True  elapsed=2.0min
[PPP linear | lambda=1.0 | gamma=0.003] win 08/10  test 25-01~25-03  obj=-0.003648  success=True  elapsed=2.3min
[PPP linear | lambda=1.0 | gamma=0.003] wi

...

,method,lambda,gamma,Ann_Ret,Ann_Vol,Sharpe,Sortino,MDD,Win,AvgTO
0,PPP,5.0,0.003,0.099368,0.173275,0.573474,1.877401,-0.138021,0.566667,0.228335



Running PPP linear baseline: lambda=10.0, gamma=0.003
[PPP linear | lambda=10.0 | gamma=0.003] win 01/10  test 23-04~23-06  obj=+0.061085  success=True  elapsed=0.4min
[PPP linear | lambda=10.0 | gamma=0.003] win 02/10  test 23-07~23-09  obj=+0.063951  success=True  elapsed=0.6min
[PPP linear | lambda=10.0 | gamma=0.003] win 03/10  test 23-10~23-12  obj=+0.065740  success=True  elapsed=0.9min
[PPP linear | lambda=10.0 | gamma=0.003] win 04/10  test 24-01~24-03  obj=+0.068252  success=True  elapsed=1.2min
[PPP linear | lambda=10.0 | gamma=0.003] win 05/10  test 24-04~24-06  obj=+0.063972  success=True  elapsed=1.5min
[PPP linear | lambda=10.0 | gamma=0.003] win 06/10  test 24-07~24-09  obj=+0.059867  success=True  elapsed=1.8min
[PPP linear | lambda=10.0 | gamma=0.003] win 07/10  test 24-10~24-12  obj=+0.060585  success=True  elapsed=2.1min
[PPP linear | lambda=10.0 | gamma=0.003] win 08/10  test 25-01~25-03  obj=+0.059238  success=True  elapsed=2.2min
[PPP linear | lambda=10.0 | gamma

,method,lambda,gamma,Ann_Ret,Ann_Vol,Sharpe,Sortino,MDD,Win,AvgTO
0,PPP,10.0,0.003,0.094475,0.170965,0.552599,1.783885,-0.133713,0.566667,0.194319



PPP lambda sweep summary:


,method,lambda,gamma,Ann_Ret,Ann_Vol,Sharpe,Sortino,MDD,Win,AvgTO
0,PPP,0.1,0.003,0.297382,0.282384,1.053111,2.274627,-0.186816,0.600000,0.635446
1,PPP,0.5,0.003,0.272896,0.264275,1.032618,2.265719,-0.176580,0.600000,0.590013
2,PPP,1.0,0.003,0.218910,0.228146,0.959519,2.161780,-0.158091,0.566667,0.503483
3,PPP,2.0,0.003,0.131134,0.185483,0.706985,1.927838,-0.145662,0.533333,0.345373
4,PPP,5.0,0.003,0.099368,0.173275,0.573474,1.877401,-0.138021,0.566667,0.228335
5,PPP,10.0,0.003,0.094475,0.170965,0.552599,1.783885,-0.133713,0.566667,0.194319


In [7]:
# =========================
# 6. 可选：保存小 summary；默认不保存权重文件
# =========================
SAVE_SUMMARY_CSV = True
OUT_DIR = DATA_DIR / "ppp_outputs"
OUT_DIR.mkdir(exist_ok=True)

if SAVE_SUMMARY_CSV:
    mse_df.to_csv(OUT_DIR / "mse_alignment_summary.csv", index=False, encoding="utf-8-sig")
    ppp_summary.to_csv(OUT_DIR / "ppp_lambda_sweep_summary.csv", index=False, encoding="utf-8-sig")
    print("已保存 summary 到：", OUT_DIR)


已保存 summary 到： D:\科研和竞赛\ppp_outputs
